# Multimodal (Vision-Language) — โจทย์แบบ "รูป -> ข้อความ"

เช่น บรรยายรูปเป็นภาษาไทย (image captioning), ตอบคำถามจากรูป (VQA), อ่านตัวอักษรในรูป (OCR)

วิธีในโน้ตบุ๊กนี้ (สำหรับ image captioning ไทย):
- วิธีที่ 1 (ง่ายสุด มือใหม่ทำแค่นี้) = ใช้โมเดลที่เทรนกับ Thai-COCO มาแล้ว generate ได้เลย ไม่ต้องเทรน
- วิธีที่ 2 (ไม่บังคับ) = `BLIP` fine-tune (เบา)
- วิธีที่ 3 (ขั้นสูง ต้อง GPU 16GB) = `PaliGemma2` + LoRA คะแนนดีสุด


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ input มีรูป และ output เป็น "ข้อความ" (ประโยค/คำตอบ)
ถ้า output เป็น "1 คลาส" -> ไปหัวข้อ 01 Computer Vision

สำคัญมาก: metric คือ `BLEU` ที่ต้องตัดคำไทยด้วย `newmm` ก่อน
ใช้ฟังก์ชัน `thai_bleu()` ในโน้ตบุ๊กวัดคะแนนในเครื่องก่อนส่งทุกครั้ง
ต้องแก้ (`# << แก้`): ชื่อ competition, path โฟลเดอร์รูป/ไฟล์ JSON

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U transformers accelerate datasets pillow pandas torch sentencepiece
!pip -q install pythainlp nltk          # ใช้วัด Thai BLEU
# !pip -q install peft bitsandbytes     # ปลดคอมเมนต์เฉพาะถ้าจะทำวิธีที่ 3 (PaliGemma2 LoRA) -- มือใหม่ไม่ต้องลง

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "super-ai-engineer-ss-6-thai-language-image-captioning"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — path + ตัววัด Thai BLEU (นี่คือ metric จริง)

In [ ]:
import os, glob, json, pandas as pd, torch
from PIL import Image
TRAIN_IMG  = os.path.join(DATA_DIR,"train")     # << แก้: ให้ตรงโฟลเดอร์รูป train เช่นถ้าเห็น "images/train" ใช้ os.path.join(DATA_DIR,"images","train")
TEST_IMG   = os.path.join(DATA_DIR,"test")      # << แก้: ให้ตรงโฟลเดอร์รูป test เช่น os.path.join(DATA_DIR,"images","test")
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")  # << แก้: ถ้าไฟล์ไม่ได้ชื่อนี้ ดูชื่อจริงจากผลเซลล์ดาวน์โหลด
js = glob.glob(os.path.join(DATA_DIR,"*train*.json")); TRAIN_JSON = js[0] if js else None
sample = pd.read_csv(SAMPLE_SUB); ID_COL, CAP_COL = sample.columns[0], sample.columns[1]
print("รูป test:", TEST_IMG, "| sample คอลัมน์:", list(sample.columns)); display(sample.head())

# BLEU = วัดว่าประโยคที่โมเดลเขียน เหมือนเฉลยแค่ไหน (0-1 ยิ่งสูงยิ่งดี) ; ไทยต้องตัดคำด้วย newmm ก่อน ไม่งั้นคะแนนเพี้ยน
from pythainlp.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction   # ไม่ต้อง nltk.download อะไร
def th_tok(s): return [w for w in word_tokenize(str(s),engine="newmm",keep_whitespace=False) if w.strip()]
def thai_bleu(list_of_refs, hyps):
    refs=[[th_tok(r) for r in rs] for rs in list_of_refs]; hyp=[th_tok(h) for h in hyps]
    return corpus_bleu(refs,hyp,weights=(0.25,0.25,0.25,0.25),smoothing_function=SmoothingFunction().method4)
def write_submission(pred_dict, out="submission.csv"):
    def look(idv):
        s=str(idv)
        # ลองหลายแบบ: ตรง ๆ / เติม .jpg / เอาเฉพาะชื่อไฟล์ / ตัดนามสกุลออก
        for k in (s, s+".jpg", os.path.basename(s), os.path.splitext(os.path.basename(s))[0]):
            if k in pred_dict: return pred_dict[k]
        return "ภาพ"   # fallback ไทยสั้น ๆ กัน BLEU=0
    o=sample.copy(); o[CAP_COL]=o[ID_COL].map(look)
    n_fb=(o[CAP_COL]=="ภาพ").sum()
    if n_fb: print("เตือน: เติม fallback", n_fb, "แถว (ถ้าเยอะแปลว่า id ไม่ตรง -> เช็ค ID_COL/นามสกุลไฟล์)")
    o.to_csv(out,index=False,encoding="utf-8-sig"); print("บันทึก",out,o.shape); return o

## วิธีที่ 1 — โมเดล Thai-COCO สำเร็จรูป (ง่ายสุด ไม่ต้องเทรน)

ใช้โมเดลที่เคยเทรนกับคลังข้อมูลเดียวกัน generate คำบรรยายได้เลยใน <1 ชม.

In [ ]:
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
M="Natthaphon/thaicapgen-convnext-phayathai"   # << แก้โมเดลได้
dev="cuda" if torch.cuda.is_available() else "cpu"
fe=AutoImageProcessor.from_pretrained(M); tk=AutoTokenizer.from_pretrained(M)
model=AutoModel.from_pretrained(M,trust_remote_code=True).to(dev).eval()
pred={}
for p in sorted(glob.glob(os.path.join(TEST_IMG,"*"))):
    px=fe(images=[Image.open(p).convert("RGB")],return_tensors="pt").pixel_values.to(dev)
    with torch.no_grad(): ids=model.generate(px,max_length=120,num_beams=4)   # ถ้าไม่มี GPU จะช้ามาก แนะนำเปิด GPU
    pred[os.path.basename(p)]=tk.batch_decode(ids,skip_special_tokens=True)[0].strip()
print("ตัวอย่างคำบรรยายที่ได้:", list(pred.items())[:2])   # ควรเห็นข้อความไทย เช่น 'ผู้ชายกำลังเล่นเซิร์ฟ'
write_submission(pred,"submission.csv")

## ตรวจคะแนน BLEU ในเครื่องก่อนส่ง (สำคัญ! ทำทุกครั้ง)

เอารูปฝั่ง train (ที่รู้คำเฉลย) มาลองทำนายแล้วเรียก `thai_bleu()` ดูคะแนน ถ้าต่ำมาก (ใกล้ 0) แปลว่ามีอะไรผิด (path/id) อย่าเพิ่งส่ง

In [ ]:
def load_refs(json_path, img_dir, limit=50):
    if not json_path: return []
    data=json.load(open(json_path,encoding="utf-8")); pairs={}
    def add(fn,cap):
        if not fn or not cap: return
        p=os.path.join(img_dir, os.path.basename(str(fn)))
        if os.path.exists(p): pairs.setdefault(p,[]).append(str(cap).strip())
    if isinstance(data,dict) and "annotations" in data:
        id2n={im["id"]:im.get("file_name") or im.get("filename") for im in data.get("images",[])}
        for a in data["annotations"]: add(id2n.get(a.get("image_id")), a.get("caption") or a.get("th_caption"))
    elif isinstance(data,dict):
        for k,v in data.items():
            for c in (v if isinstance(v,list) else [v]): add(k, c if isinstance(c,str) else (c.get("caption") if isinstance(c,dict) else None))
    elif isinstance(data,list):
        for r in data: add(r.get("image") or r.get("file_name"), r.get("caption") or r.get("th_caption"))
    return list(pairs.items())[:limit]
val=load_refs(TRAIN_JSON, TRAIN_IMG, 50)
if val:
    hyps=[]; refs=[]
    for p,caps in val:
        px=fe(images=[Image.open(p).convert("RGB")],return_tensors="pt").pixel_values.to(dev)
        with torch.no_grad(): ids=model.generate(px,max_length=120,num_beams=4)
        hyps.append(tk.batch_decode(ids,skip_special_tokens=True)[0].strip()); refs.append(caps)
    print("Thai BLEU บน train (ลอง 50 รูป) =", round(thai_bleu(refs,hyps),4), "(ยิ่งสูงยิ่งดี; ใกล้ 0 = มีอะไรผิด อย่าเพิ่งส่ง)")
else:
    print("โหลด reference จาก TRAIN_JSON ไม่ได้ -> ข้ามการวัดในเครื่อง (เช็ค path/รูปแบบ JSON)")

## วิธีที่ 2 — BLIP fine-tune (ไม่บังคับ)

โค้ดเต็มอยู่ใน reference_cheatsheet.md (โหลดคู่รูป-คำบรรยายจาก JSON มาเทรนต่อ)
มือใหม่ข้ามได้ ใช้วิธีที่ 1 ก็ได้คะแนนแล้ว

## วิธีที่ 3 — PaliGemma2 + LoRA (ขั้นสูง ต้อง GPU 16GB คะแนนดีสุด)

โค้ดเต็มอยู่ใน reference_cheatsheet.md หัวข้อ D (prompt `caption th` + LoRA 4-bit)

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "thai coco pretrained captioning"
!kaggle competitions submissions -c {COMP} | head